# 01 - Exploratory data analysis

**Status: not yet run on real data.** This notebook has no outputs because the Olist dataset has not been
downloaded yet. Run `make data` first, then execute every cell. Until then, nothing here is a finding.

All numbers come from `pipelines.eda` (unit-tested). The notebook only adds plots. After a real run, copy the
confirmed facts into `docs/DATA_PLAN.md` section 14.

In [ ]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import pandas as pd

from core.settings import get_settings
from pipelines import eda

settings = get_settings()
WAREHOUSE = settings.duckdb_path          # data/warehouse/warehouse.duckdb
con = duckdb.connect(str(WAREHOUSE), read_only=True)
synthetic = con.execute("SELECT value FROM meta.build WHERE key = 'synthetic'").fetchone()[0] == "true"
assert not synthetic, "This warehouse was built from FAKE data. Rebuild with `make data` on the real dataset."

In [ ]:
findings = eda.run(WAREHOUSE, settings.reports_dir)
window = findings["monitoring_window"]["window"]
window

## 1. Date coverage and the monitoring window

In [ ]:
daily = con.execute(
    "SELECT d.date, count(o.order_id) AS orders FROM marts.dim_date d "
    "LEFT JOIN marts.fct_orders o ON o.purchase_date = d.date "
    "WHERE d.date <= (SELECT max(purchase_date) FROM marts.fct_orders) GROUP BY 1 ORDER BY 1"
).df()
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(daily["date"], daily["orders"], lw=0.8)
ax.axvspan(pd.Timestamp(window["start"]), pd.Timestamp(window["end"]), alpha=0.12, label="dense window")
ax.axvline(pd.Timestamp(findings["monitoring_window"]["first_detection_date"]), ls="--", c="k", label="first detection date")
ax.set(title="Orders per day", ylabel="orders"); ax.legend(); plt.show()

## 2. Seasonality

In [ ]:
s = findings["seasonality"]
pd.Series(s["weekday_index"]).reindex(["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]).plot.bar(title="Weekday index (1.0 = average day)")
plt.axhline(1.0, c="k", lw=0.6); plt.show()
{k: s[k] for k in ("holiday_ratio", "black_friday_2017_ratio_to_prior_3_weeks", "strongest_weekday_effect")}

## 3. KPI time series and distributions

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(12, 10), sharex=True)
for ax, (kpi, sql) in zip(axes.ravel(), eda.KPI_SQL.items()):
    frame = con.execute(f"SELECT * FROM ({sql}) WHERE date BETWEEN ? AND ? ORDER BY date", [window["start"], window["end"]]).df()
    ax.plot(frame["date"], frame["value"], lw=0.7); ax.set_title(kpi, fontsize=9)
plt.tight_layout(); plt.show()
pd.DataFrame(findings["kpi_distributions"]).T

## 4. Segments and the minimum-sample rule

In [ ]:
rows = [
    {"dimension": dim, "values": d["cardinality"], "expected": d["expected_cardinality"], "kind": kind, **counts}
    for dim, d in findings["segments"]["dimensions"].items()
    for kind, counts in d["kinds"].items()
]
pd.DataFrame(rows)

## 5. Expected data issues (DATA_PLAN.md section 3)

In [ ]:
pd.DataFrame(findings["expected_issues"])